# Proyecto final · 02 · Silver

**Columna vertebral, parte 2 · Sección 3 del examen.** La tabla silver es la versión limpia de bronze.

Lo que se evalúa:
- Tipos correctos (fechas como fecha, montos como número).
- Nulos tratados con una regla que puedas explicar.
- Deduplicación por la llave del negocio.
- Al menos un join que enriquezca la tabla.
- Una celda de métricas antes y después.

In [0]:
-- Seleccionar el catálogo del proyecto para trabajar con sus esquemas y tablas.
USE CATALOG pf_fallas;

In [0]:
-- TU CÓDIGO ACÁ

-- Crear una tabla de referencia con los dos tipos de vehículos eléctricos.
-- Esta tabla permite agregar la sigla y la descripción mediante un JOIN en Silver.

CREATE OR REPLACE TABLE silver.tipos_vehiculo
USING DELTA AS
SELECT *
FROM VALUES
    (
        'Battery Electric Vehicle (BEV)',
        'BEV',
        'Vehículo totalmente eléctrico'
    ),
    (
        'Plug-in Hybrid Electric Vehicle (PHEV)',
        'PHEV',
        'Vehículo híbrido enchufable'
    )
AS tipos(electric_vehicle_type, type_code, type_description);

num_affected_rows,num_inserted_rows


In [0]:
-- TU CÓDIGO 
-- Crear la tabla Silver: convertir tipos, tratar valores vacíos y conservar
-- una sola fila por vehicle_id, priorizando la primera ingesta.
-- Agregar la sigla y la descripción del tipo de vehículo mediante un LEFT JOIN.
-- Conservar la autonomía original y crear una columna para promedios sin ceros.

CREATE OR REPLACE TABLE silver.vehiculos
USING DELTA AS
WITH limpieza AS (
    SELECT
        TRY_CAST(vehicle_id AS BIGINT) AS vehicle_id,
        TRIM(vin_prefix) AS vin_prefix,

        COALESCE(NULLIF(TRIM(county), ''), 'SIN DATO') AS county,
        COALESCE(NULLIF(TRIM(city), ''), 'SIN DATO') AS city,
        UPPER(TRIM(state)) AS state,

        NULLIF(TRIM(postal_code), '') AS postal_code,
        TRY_CAST(model_year AS INT) AS model_year,
        UPPER(TRIM(make)) AS make,
        TRIM(model) AS model,

        TRIM(electric_vehicle_type) AS electric_vehicle_type,
        TRIM(cafv_eligibility) AS cafv_eligibility,

        TRY_CAST(electric_range AS INT) AS electric_range,
        CASE
            WHEN TRY_CAST(electric_range AS INT) > 0
            THEN TRY_CAST(electric_range AS INT)
            ELSE NULL
        END AS electric_range_valid,

        TRY_CAST(legislative_district AS INT) AS legislative_district,
        NULLIF(TRIM(vehicle_location), '') AS vehicle_location,
        NULLIF(TRIM(electric_utility), '') AS electric_utility,
        NULLIF(TRIM(census_tract_2020), '') AS census_tract_2020,

        archivo_origen,
        fecha_ingesta
    FROM bronze.vehiculos_raw
),
deduplicacion AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY vehicle_id
            ORDER BY fecha_ingesta ASC, archivo_origen ASC
        ) AS numero_fila
    FROM limpieza
    WHERE vehicle_id IS NOT NULL
)
SELECT
    v.* EXCEPT (numero_fila),
    t.type_code,
    t.type_description
FROM deduplicacion v
LEFT JOIN silver.tipos_vehiculo t
    ON v.electric_vehicle_type = t.electric_vehicle_type
WHERE v.numero_fila = 1;

num_affected_rows,num_inserted_rows


### Métricas antes y después
Ajusta las columnas a tu dataset: la llave del negocio y las columnas donde importa que no haya nulos.

In [0]:
-- Comparar Bronze y Silver para verificar el total de filas, los vehículos
-- únicos, los registros repetidos y las ciudades con valores nulos o vacíos.
-- Esta comparación muestra el efecto de la limpieza y la deduplicación.

SELECT
    'bronze' AS capa,
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT vehicle_id) AS vehiculos_unicos,
    COUNT(*) - COUNT(DISTINCT vehicle_id) AS registros_repetidos,
    COUNT_IF(city IS NULL OR TRIM(city) = '') AS ciudades_sin_tratar
FROM bronze.vehiculos_raw

UNION ALL

SELECT
    'silver' AS capa,
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT vehicle_id) AS vehiculos_unicos,
    COUNT(*) - COUNT(DISTINCT vehicle_id) AS registros_repetidos,
    COUNT_IF(city IS NULL OR TRIM(city) = '') AS ciudades_sin_tratar
FROM silver.vehiculos;

capa,filas_totales,vehiculos_unicos,registros_repetidos,ciudades_sin_tratar
bronze,30010,30000,10,4
silver,30000,30000,0,0


In [0]:
-- Verificar que todos los vehículos encontraron su tipo en la tabla
-- de referencia y revisar cuántos corresponden a BEV y PHEV.
SELECT
    type_code,
    type_description,
    COUNT(*) AS total_vehiculos
FROM pf_fallas.silver.vehiculos
GROUP BY type_code, type_description
ORDER BY type_code;

type_code,type_description,total_vehiculos
BEV,Vehículo totalmente eléctrico,23780
PHEV,Vehículo híbrido enchufable,6220


### TU RESPUESTA
¿Qué regla usaste para los nulos y los duplicados, y por qué esa y no otra?

Los valores vacíos de `city` y `county` se reemplazaron por `SIN DATO`
para conservar los vehículos sin inventar una ubicación. Los demás nulos
se conservaron cuando no había información confiable para completarlos.

La deduplicación se realizó con `ROW_NUMBER()`, usando `vehicle_id`
como llave de negocio y conservando la primera ingesta. Se eliminaron
`10` registros repetidos y se mantuvieron los `30.000` vehículos únicos.

La autonomía original se conservó. También se creó `electric_range_valid`,
que deja como `NULL` los valores iguales o menores que cero para excluirlos
de los promedios.

Finalmente, un `LEFT JOIN` agregó la sigla y la descripción del tipo
de vehículo sin descartar registros que no encontraran una coincidencia.